# Causal CWT Walkthrough — and How Well It Approximates the Full CWT

Two questions:

1. **What do `_ricker_causal` and `causal_cwt` actually compute?** Walk through the kernel shape, the causal rolling z-normalization, and the FFT convolution, and verify strict causality on a synthetic signal.
2. **How good is the causal CWT at guessing the full (two-sided) CWT?** Compute both on real Nasdaq prices and quantify the gap per scale: pointwise correlation, optimal lag, and edge behavior at the latest bar — which is the only bar that matters for live trading.

**Setup:** `_ricker_causal(s, n)` is one-sided on `t ∈ [-4s, 0]`; the textbook Ricker is symmetric on `t ∈ [-4s, +4s]`. The causal version is the same kernel with the right half (`t > 0`) zeroed out — that's why we expect it to lag the symmetric version by roughly the right-half mass center, i.e. `~2s` samples.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import fftconvolve

from ss_wavelets import causal_cwt, ALL_SCALES
from ss_wavelets.cwt import _ricker_causal
from ss_loaders import load_price_matrix
from ss_plotting import plot_scalogram_heatmap

np.set_printoptions(suppress=True, precision=4)

## 1. The Ricker (Mexican-hat) wavelet — symmetric vs. one-sided

Symmetric Ricker on `t ∈ [-4s, +4s] / s`:
$$\psi(t) = \frac{1}{\sqrt{s}}\,(1 - t^2)\,\exp(-t^2/2)$$

`_ricker_causal` keeps only the left half (`t ≤ 0`). The right half — which would peek into the future when convolved — is discarded. Plot both to see the asymmetry.

In [ ]:
def ricker_symmetric(scale: int) -> np.ndarray:
    """Standard two-sided Ricker on t in [-4*scale, +4*scale]."""
    points = 4 * scale
    t = np.arange(-points, points + 1) / scale
    return (1.0 - t ** 2) * np.exp(-t ** 2 / 2.0) / np.sqrt(scale)


demo_scales = [5, 21, 90]
fig, axes = plt.subplots(1, len(demo_scales), figsize=(13, 3.5), sharey=True)
for ax, s in zip(axes, demo_scales):
    sym = ricker_symmetric(s)
    cau = _ricker_causal(s, n_dates=10_000)  # n_dates large enough not to clip
    sym_t = np.arange(-4 * s, 4 * s + 1)
    cau_t = np.arange(-(len(cau) - 1), 1)
    ax.plot(sym_t, sym, color='tab:blue', label='symmetric Ricker')
    ax.plot(cau_t, cau, color='tab:red',  label='_ricker_causal', lw=2)
    ax.axvline(0, color='k', lw=0.5)
    ax.set_title(f'scale = {s} days')
    ax.set_xlabel('t (days, 0 = now)')
    ax.grid(alpha=0.3)
axes[0].set_ylabel(r'$\psi(t)$')
axes[0].legend(loc='lower left', fontsize=9)
plt.suptitle('Ricker kernels — symmetric vs. causal (one-sided)', y=1.02)
plt.tight_layout(); plt.show()

Notice three things:

- Causal kernel is the **left half** of the symmetric one. The right lobe (t > 0) is gone.
- Both have a peak at t = 0 — the right edge of the causal kernel is its largest value, so the most recent price gets the strongest weight in the convolution.
- The kernel width grows linearly with `scale`. At scale = 90, the causal kernel reaches ~360 days into the past.

## 2. `causal_cwt` step-by-step on a synthetic signal

Build a deterministic price series so we can verify strict causality. We'll inject a sharp *future* spike at t=600 and confirm that `causal_cwt` output at t=599 is unchanged whether the spike is there or not.

In [ ]:
rng = np.random.default_rng(0)
n_dates = 1000
t = np.arange(n_dates)
trend = 100 + 0.05 * t                                 # slow upward drift
cycle = 5 * np.sin(2 * np.pi * t / 60)                 # 60-day cycle
noise = rng.normal(0, 0.5, size=n_dates).cumsum() * 0.3
synthetic = (trend + cycle + noise)[:, None]           # (n_dates, 1)

# Two versions: identical up to t=600, differ only at and after t=600
syn_a = synthetic.copy()
syn_b = synthetic.copy()
syn_b[600:610] += 30                                   # large future spike

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(syn_a[:, 0], label='series A (no spike)', lw=1)
ax.plot(syn_b[:, 0], label='series B (spike at t=600)', lw=1, alpha=0.8)
ax.axvline(600, color='k', ls='--', lw=0.8)
ax.set_title('Synthetic price series — identical for t < 600'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 2a. Causal rolling z-normalization

Before convolution, each series is z-normalized using a *causal* rolling window of length `lookback`. The cumulative-sum trick makes this O(n_dates) per ticker:

```python
cs  = np.cumsum([0, x[0], x[1], ...])           # prefix sum of x
cs2 = np.cumsum([0, x[0]**2, x[1]**2, ...])     # prefix sum of x^2
lo  = max(0, t - lookback + 1)                  # left edge of window
mu  = (cs[t+1] - cs[lo]) / (t - lo + 1)         # mean over [lo, t]
std = sqrt(mu2 - mu**2)
```

Crucially, `lo` and `t+1` are both ≤ t+1 — only past-and-present prices contribute. Plot the rolling mean ± std band to see the window in action.

In [ ]:
lookback = 120
prices = syn_a
n, _ = prices.shape

cs  = np.cumsum(np.vstack([np.zeros((1, 1)), prices]),       axis=0)
cs2 = np.cumsum(np.vstack([np.zeros((1, 1)), prices ** 2]),  axis=0)
idx = np.arange(n)
lo  = np.maximum(0, idx - lookback + 1)
counts = (idx - lo + 1)[:, None]
mu  = (cs[idx + 1] - cs[lo]) / counts
mu2 = (cs2[idx + 1] - cs2[lo]) / counts
std = np.sqrt(np.maximum(mu2 - mu ** 2, 1e-4))
x_norm = (prices - mu) / std

fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
axes[0].plot(prices[:, 0], color='tab:blue', label='price')
axes[0].plot(mu[:, 0], color='tab:orange', label=f'rolling mean ({lookback}d)')
axes[0].fill_between(idx, (mu - std)[:, 0], (mu + std)[:, 0], color='tab:orange', alpha=0.15, label='±1 std')
axes[0].legend(); axes[0].set_title('Causal rolling stats'); axes[0].grid(alpha=0.3)
axes[1].plot(x_norm[:, 0], color='tab:green')
axes[1].axhline(0, color='k', lw=0.5)
axes[1].set_title('z-normalized series (input to convolution)'); axes[1].grid(alpha=0.3)
axes[1].set_xlabel('t')
plt.tight_layout(); plt.show()

### 2b. The convolution slice — `full[:T]` is the causal output

`fftconvolve(x_norm, kernel, mode='full')` produces an array of length `n_dates + len(kernel) - 1`. Each output index `t` of the full convolution is
$$\text{full}[t] = \sum_k x[k]\, K[t - k]$$

where `K` runs over `[-points, 0]`. So `full[t]` only uses `x[k]` with `k ≤ t` — i.e. strictly past-and-present samples. Slicing `full[:n_dates]` aligns the right edge of the kernel with index t at every step.

Run `causal_cwt` on both series and verify: outputs at `t < 600` are bit-identical.

In [ ]:
scales_demo = [5, 21, 90]
cwt_a = causal_cwt(syn_a, scales_demo, lookback=120)  # (n_scales, n_dates, 1)
cwt_b = causal_cwt(syn_b, scales_demo, lookback=120)

# Strict causality check — outputs must agree everywhere t < 600
diff_pre  = np.max(np.abs(cwt_a[:, :600] - cwt_b[:, :600]))
diff_post = np.max(np.abs(cwt_a[:, 600:] - cwt_b[:, 600:]))
print(f'max |Δ| for t <  600 (must be 0):     {diff_pre:.3e}')
print(f'max |Δ| for t >= 600 (should be > 0): {diff_post:.3e}')
assert diff_pre == 0.0, 'causality violation detected'
print('strict causality holds.')

In [ ]:
fig, axes = plt.subplots(len(scales_demo), 1, figsize=(11, 7), sharex=True)
for ax, s, ca, cb in zip(axes, scales_demo, cwt_a, cwt_b):
    ax.plot(ca[:, 0], color='tab:blue', label='series A')
    ax.plot(cb[:, 0], color='tab:red',  label='series B', alpha=0.7)
    ax.axvline(600, color='k', ls='--', lw=0.8)
    ax.set_ylabel(f'scale={s}')
    ax.grid(alpha=0.3)
axes[0].set_title('Causal CWT response — coincides until the spike, diverges only after')
axes[0].legend(loc='upper left')
axes[-1].set_xlabel('t')
plt.tight_layout(); plt.show()

## 3. The non-causal "full" CWT — what we'd compute if we had future data

Same z-normalization (causal — keeps the comparison about *kernel asymmetry only*), but now the kernel is the **symmetric** Ricker centered at t. The convolution is sliced so output[t] aligns with the *center* of the kernel, which means it uses both `x[t-4s, ..., t]` (past) and `x[t, ..., t+4s]` (future).

This is the textbook CWT — and it requires future data, so it cannot be used in real-time trading. It's the "oracle" that `causal_cwt` is trying to approximate.

In [ ]:
def full_cwt(prices: np.ndarray, scales: list[int], lookback: int) -> np.ndarray:
    """Two-sided (non-causal) CWT — output[t] uses x[t-4s : t+4s].

    Identical causal z-norm to `causal_cwt`, but with the symmetric Ricker
    kernel and a 'same'-style alignment so we isolate the effect of kernel
    asymmetry. The output at the right edge has truncated kernel support
    (no future data to convolve against), so we leave it as a NaN strip
    of width `4*scale` to make the missing-future region explicit.
    """
    n_dates, n_tickers = prices.shape

    cs  = np.cumsum(np.vstack([np.zeros((1, n_tickers)), prices]),      axis=0)
    cs2 = np.cumsum(np.vstack([np.zeros((1, n_tickers)), prices ** 2]), axis=0)
    idx = np.arange(n_dates)
    lo  = np.maximum(0, idx - lookback + 1)
    counts = (idx - lo + 1)[:, None]
    mu  = (cs[idx + 1] - cs[lo]) / counts
    mu2 = (cs2[idx + 1] - cs2[lo]) / counts
    std = np.sqrt(np.maximum(mu2 - mu ** 2, 1e-4))
    x_norm = (prices - mu) / std

    coeffs = np.zeros((len(scales), n_dates, n_tickers), dtype=np.float32)
    for si, s in enumerate(scales):
        kernel = ricker_symmetric(s).astype(np.float32)
        full = fftconvolve(x_norm, kernel[:, None], mode='same', axes=0)
        # Mask the right strip where the kernel hangs off the end of x
        coeffs[si] = full.astype(np.float32)
        coeffs[si, n_dates - 4 * s:] = np.nan
    return coeffs

## 4. Side-by-side scalograms on real Nasdaq data

Load a slice of the Nasdaq3347 universe, pick one ticker, and render both scalograms via `ss_plotting.plot_scalogram_heatmap` — the same helper the rest of the workspace uses. It takes **power** (`|coeff|²`), which is also what `precompute_windows` and the regime divergence consume downstream, so this visualization matches the form the pipeline actually flows.

In [ ]:
prices_df, _, _ = load_price_matrix(
    './Nasdaq3347',
    min_history=504,
    start_date='2018-01-01',
    end_date='2024-12-31',
)
# Pick one ticker for visualization
ticker = 'AAPL' if 'AAPL' in prices_df.columns else prices_df.columns[0]
px = prices_df[[ticker]].values
print(f'using {ticker}, {px.shape[0]} dates')

scales = ALL_SCALES
lookback = 120
cwt_causal = causal_cwt(px, scales, lookback=lookback)
cwt_full   = full_cwt(px,   scales, lookback=lookback)
print('causal:', cwt_causal.shape, '| full:', cwt_full.shape)

In [ ]:
scales_arr = np.array(scales)
dates = prices_df.index.values

# Power = |coeff|^2 — what `precompute_windows` and the regime divergence consume.
# Full CWT has NaN on the right edge (no future data); plot_scalogram_heatmap
# applies its own +1e-12 epsilon so the helper handles zeros and NaNs cleanly.
causal_power = (cwt_causal[:, :, 0] ** 2).astype(np.float64)
full_power   = (cwt_full[:, :, 0]   ** 2).astype(np.float64)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
plot_scalogram_heatmap(
    causal_power, scales_arr, dates,
    title=f'{ticker} — causal_cwt power (one-sided kernel, no future leakage)',
    ax=axes[0],
)
plot_scalogram_heatmap(
    full_power, scales_arr, dates,
    title=f'{ticker} — full CWT power (symmetric kernel, uses future)',
    ax=axes[1],
)
axes[1].set_xlabel('date')
plt.tight_layout(); plt.show()

Read the two panels together:

- Both scalograms light up in the same date × scale regions — volatility flares, multi-week trends.
- The causal panel's features are visibly **shifted to the right** vs. the full panel; the shift grows with scale. That's the cost of refusing to look ahead.
- The right strip of the full panel is dark because it's NaN — the symmetric kernel would need future data the dataset doesn't have.

The visual gap is suggestive. The lag analysis below quantifies it.

## 5. Quantifying the gap — correlation per scale and optimal lag

The kernel asymmetry should produce a roughly fixed lag: the causal kernel's mass center is around `t = -2*scale` (because the half-kernel runs from `-4s` to `0`). So we'd expect `causal[t] ≈ full[t - 2s]`.

Test this directly by sweeping shifts `k` and computing the correlation between `causal[t]` and `full[t - k]` for each scale. The optimal `k` and the achievable max correlation tell us how good the causal version is at "guessing" the full version after we account for the inevitable lag.

In [ ]:
def best_lag_correlation(causal_1d: np.ndarray, full_1d: np.ndarray, max_lag: int):
    """For each lag k in [0, max_lag], correlate causal[t] vs full[t-k].

    Returns (lags, corrs). Drops NaN rows (right edge of full).
    """
    lags = np.arange(0, max_lag + 1)
    corrs = np.empty(len(lags))
    for i, k in enumerate(lags):
        a = causal_1d[k:]              # causal[t] for t >= k
        b = full_1d[: len(full_1d) - k]  # full[t-k] aligned
        m = np.isfinite(a) & np.isfinite(b)
        if m.sum() < 100:
            corrs[i] = np.nan
        else:
            corrs[i] = np.corrcoef(a[m], b[m])[0, 1]
    return lags, corrs


max_lag = 4 * max(scales)  # cover up to the largest expected shift (~2*scale)
rows = []
fig, ax = plt.subplots(figsize=(10, 5))
for si, s in enumerate(scales):
    lags, corrs = best_lag_correlation(cwt_causal[si, :, 0], cwt_full[si, :, 0], max_lag)
    best_k = int(lags[np.nanargmax(corrs)])
    best_r = float(np.nanmax(corrs))
    zero_r = float(corrs[0])
    rows.append((s, zero_r, best_k, best_r, 2 * s))
    ax.plot(lags, corrs, label=f'scale={s}')
ax.set_xlabel('lag k applied to full (causal[t] vs full[t-k])')
ax.set_ylabel('correlation')
ax.set_title('Cross-correlation of causal vs. lagged full CWT, per scale')
ax.grid(alpha=0.3); ax.legend(ncol=2, fontsize=8)
plt.tight_layout(); plt.show()

summary = pd.DataFrame(rows, columns=['scale', 'corr@lag=0', 'best_lag', 'corr@best_lag', 'expected_lag (2*scale)'])
summary

Read the summary table top-to-bottom:

- **`corr@lag=0`** is how well the causal CWT directly predicts the full CWT *at the same time index*. It drops sharply with scale — at scale 90+, the causal value at time t bears almost no relation to what an oracle (with future data) would say at the same instant.
- **`best_lag`** is the optimal shift; it tracks `2*scale` closely. That's the price of refusing to look ahead — the causal CWT's response to a feature shows up `~2s` days later than the symmetric CWT's response.
- **`corr@best_lag`** is the achievable correlation once you grant the causal version that lag. It stays high (typically > 0.9) across all scales, meaning the causal CWT *does* recover the full CWT — but a delayed copy of it.

Translation for live trading: the causal CWT at time t is not an estimate of the *current* full CWT; it's an honest estimate of what the full CWT would have said `~2*scale` days ago.

## 6. Edge behavior — what does the causal CWT see at the latest bar?

In live trading only the *last column* of the scalogram is queried. Compare `cwt_causal[:, T-1]` against the full CWT at progressively earlier dates `T-1-k`. The causal value at the right edge should look most like the full CWT roughly `2*scale` days back — confirming the lag interpretation at the only point that matters operationally.

In [ ]:
T = px.shape[0]
# Use a date far enough back that full CWT is defined for all needed shifts
anchor = T - 4 * max(scales) - 5

fig, axes = plt.subplots(2, 1, figsize=(11, 6))

# Panel 1: causal_cwt at anchor vs full_cwt at (anchor - 2*scale) per scale
causal_now = cwt_causal[:, anchor, 0]
full_lagged = np.array([cwt_full[si, anchor - 2 * s, 0] for si, s in enumerate(scales)])
full_same   = np.array([cwt_full[si, anchor,         0] for si, s in enumerate(scales)])

x = np.arange(len(scales))
w = 0.28
axes[0].bar(x - w, causal_now,  w, label='causal[anchor]',           color='tab:red')
axes[0].bar(x,     full_lagged, w, label='full[anchor − 2·scale]',   color='tab:blue')
axes[0].bar(x + w, full_same,   w, label='full[anchor] (oracle)',    color='tab:gray', alpha=0.6)
axes[0].set_xticks(x); axes[0].set_xticklabels(scales)
axes[0].set_xlabel('scale (days)'); axes[0].set_ylabel('coefficient')
axes[0].set_title(f'At a fixed anchor t={anchor}, the causal value matches full lagged by ~2·scale')
axes[0].axhline(0, color='k', lw=0.5); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3, axis='y')

# Panel 2: scan k=0..3*max_scale, show corr(causal_now, full[anchor-k]) per scale
ks = np.arange(0, 3 * max(scales))
for si, s in enumerate(scales):
    series = np.array([cwt_full[si, anchor - k, 0] for k in ks])
    # closest-to-causal_now distance
    dist = np.abs(series - causal_now[si])
    axes[1].plot(ks, dist, label=f's={s}')
    axes[1].axvline(2 * s, color=axes[1].lines[-1].get_color(), ls=':', alpha=0.4)
axes[1].set_xlabel('k (days back from anchor)')
axes[1].set_ylabel('|full[anchor − k] − causal[anchor]|')
axes[1].set_title('Distance between causal-now and full-k-days-ago, per scale (dotted = 2·scale)')
axes[1].legend(ncol=2, fontsize=8); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Summary

- `_ricker_causal(s, n)` is the symmetric Ricker with the future half (t > 0) chopped off. The kernel peaks at t = 0, so the most recent price always gets the largest weight.
- `causal_cwt` does two cumsum-based things — a causal rolling z-norm and an FFT convolution sliced to `full[:T]` — and the combination is strictly causal (verified by the spike-at-t=600 test).
- Compared to a textbook (symmetric-kernel) CWT that has access to future data, the causal version produces the same structural features but **lagged by roughly `2·scale` days** at each scale.
- That lag is unavoidable: it's the mass center of the right-half of the symmetric kernel that we had to discard.
- After accounting for the per-scale lag, the causal CWT recovers the full CWT with correlation typically > 0.9. So as a *delayed* estimator the approximation is quite good; as a *contemporaneous* one it's bad at long scales (which is why the regime model concentrates weight on horizons it has time to react to).